[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/17_interpretabilidad_geometria/17_proyecto_interpretabilidad_geometria.ipynb)

# Proyecto 17 · SAE sobre embeddings de Políglota

| | |
|---|---|
| **Nivel** | 🔴 Experto |
| **Duración** | ≈ 2 h |
| **GPU** | No necesaria (CPU, ≈ 3–4 min con `FAST_DEV_RUN=True`) |
| **Requiere** | Lección 17 (SAE TopK, pureza, estabilidad, steering, inversión) y módulo 15 (estadística) |
| **Conecta con** | 12 (embedders) · 15 (evaluación) · 18 (privacidad y producción) |

## 🏢 Contexto de negocio
Eres ML engineer en **Políglota Labs**. Soporte pregunta *«¿por qué el buscador cree que estos dos documentos se parecen?»* y Seguridad pregunta *«¿qué se puede reconstruir de los vectores que guardamos?»*. Tu entregable es un **informe de interpretabilidad** sobre los embeddings de Políglota: un **TopK SAE** que descompone cada vector en pocas *features* legibles, con métricas de calidad (reconstrucción, pureza, estabilidad entre semillas), una prueba de **steering** frente a una línea base y una **auditoría de filtración** del embedding.

## 📦 Dataset
Corpus Políglota (`carga_corpus`). Un SAE necesita **miles** de vectores: si el corpus es sintético usamos el generador (12 temas × 36 facetas, ≈ 2 900 docs ES/EN, con etiquetas de faceta/tema/idioma como **verdad terreno**); con datos reales (`source == "real"`) se usa `carga_corpus(fast_dev=False)` y solo el idioma como etiqueta (las métricas de pureza por faceta no aplican).
Embedder: **LSI-64** (TF-IDF + SVD) por defecto, reproducible en CPU. En Colab puedes sustituirlo por uno real (reto extra).

## 🎯 Entregables y criterios
| # | Entregable | Criterio objetivo |
|---|---|---|
| 1 | `TopKSAE.encode` (TODO 1) | Exactamente `k` no nulos por fila, valores ≥ 0, test de igualdad con `topk` |
| 2 | SAE entrenado | NMSE ≤ 0.07 con L0 = 4 y ≥ 60 latentes activos (de 96) |
| 3 | `pureza_features` (TODO 2) | Pasa el test; pureza media del SAE ≥ 0.75 y > PCA |
| 4 | `estabilidad` entre 2 semillas (TODO 3) | Fracción de átomos con cos > 0.9 calculada y reportada (≈ valor de referencia) |
| 5 | `curva_steering` (TODO 4) | α=0 → ≈ 0 % «git»; α alto → ≥ 90 %; comparación con diferencia de medias |
| 6 | Auditoría de inversión (TODO 5) | Precisión@10 de términos recuperados sin ruido ≥ 0.8; trade-off con ruido |
| 7 | Informe con recomendaciones | Cita pureza, estabilidad, baseline y riesgo de privacidad |

## 0. Preparación (celdas dadas)

In [ ]:
!pip install -q numpy scipy scikit-learn matplotlib pandas torch datasets

In [ ]:
import os, re, math, random, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
SEED, FAST_DEV_RUN = 42, True
random.seed(SEED); np.random.seed(SEED)
import torch
torch.manual_seed(SEED)
if FAST_DEV_RUN: torch.set_num_threads(1)
AZUL, NARANJA, VERDE, GRIS, ROJO = "#1F6FEB", "#F28E2B", "#2CA02C", "#6B7280", "#D62728"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25})

Utilidades del corpus Políglota (`carga_corpus`: oficial → espejo → sintético; copia autocontenida del contrato común) y retrievers de CPU.

In [ ]:
# Corpus SINTÉTICO de Políglota (último recurso, mismo esquema que el real): 12 temas x 3 facetas.
# Por faceta: términos ES (5), EN (5), identificadores de código (3). Los 3 primeros términos son "centrales"
# (salen mucho en los documentos); los 2 últimos son sinónimos raros: las queries los usan -> hueco léxico.
TEMAS = {
 "cache": (["caché", "memoria", "datos"], ["cache", "memory", "data"], [
   ("ttl", "expiración|ttl|vencimiento|caducidad|refrescar", "expiration|ttl|timeout|lifetime|refresh", "set_ttl|expire_at|touch_key"),
   ("invalidacion", "invalidación|clave|purgar|obsoleto|desalojar", "invalidation|key|purge|stale|flush", "invalidate|purge_keys|delete_pattern"),
   ("evicion", "política|lru|desalojo|capacidad|recorte", "policy|lru|eviction|capacity|trim", "lru_evict|max_entries|pop_oldest")]),
 "basedatos": (["base", "datos", "consulta"], ["database", "data", "query"], [
   ("indices", "índice|b-tree|escaneo|selectividad|acelerar", "index|btree|scan|selectivity|speedup", "create_index|explain_plan|seq_scan"),
   ("transacciones", "transacción|commit|aislamiento|rollback|atomicidad", "transaction|commit|isolation|rollback|atomicity", "begin_tx|commit_tx|rollback_tx"),
   ("migraciones", "migración|esquema|columna|versionar|alterar", "migration|schema|column|versioning|alter", "run_migration|add_column|schema_version")]),
 "auth": (["usuario", "acceso", "seguridad"], ["user", "access", "security"], [
   ("tokens", "token|jwt|firma|sesión|testigo", "token|jwt|signature|session|bearer", "verify_jwt|sign_token|refresh_token"),
   ("oauth", "oauth|permiso|consentimiento|proveedor|delegar", "oauth|scope|consent|provider|delegate", "oauth_callback|request_scope|exchange_code"),
   ("hash", "contraseña|hash|sal|bcrypt|credencial", "password|hash|salt|bcrypt|credential", "hash_password|check_password|gen_salt")]),
 "contenedores": (["contenedor", "servicio", "despliegue"], ["container", "service", "deployment"], [
   ("imagenes", "imagen|capa|dockerfile|construir|empaquetar", "image|layer|dockerfile|build|package", "docker_build|pull_image|tag_image"),
   ("redes", "red|puerto|puente|exponer|conectar", "network|port|bridge|expose|connect", "create_network|publish_port|attach_bridge"),
   ("volumenes", "volumen|montaje|persistente|disco|compartir", "volume|mount|persistent|disk|share", "mount_volume|bind_path|prune_volumes")]),
 "testing": (["prueba", "código", "calidad"], ["test", "code", "quality"], [
   ("unitarias", "unitaria|aserción|fixture|función|verificar", "unit|assertion|fixture|function|verify", "assert_equal|run_unit|make_fixture"),
   ("mocks", "simulacro|mock|dependencia|sustituir|doble", "mock|stub|dependency|replace|double", "patch_call|make_mock|reset_mock"),
   ("cobertura", "cobertura|línea|rama|informe|medir", "coverage|line|branch|report|measure", "run_coverage|branch_report|min_cover")]),
 "concurrencia": (["hilo", "tarea", "ejecución"], ["thread", "task", "execution"], [
   ("hilos", "hilo|gil|paralelo|pool|lanzar", "thread|gil|parallel|pool|spawn", "start_thread|join_all|pool_map"),
   ("asincrono", "asíncrono|corrutina|await|bucle|esperar", "async|coroutine|await|loop|wait", "run_loop|gather_tasks|await_all"),
   ("bloqueos", "bloqueo|candado|interbloqueo|exclusión|sincronizar", "lock|mutex|deadlock|exclusion|synchronize", "acquire_lock|release_lock|try_lock")]),
 "api": (["api", "petición", "recurso"], ["api", "request", "resource"], [
   ("paginacion", "paginación|cursor|página|límite|desplazamiento", "pagination|cursor|page|limit|offset", "next_cursor|page_size|fetch_page"),
   ("versionado", "versión|compatibilidad|ruta|migrar|deprecar", "version|compatibility|route|migrate|deprecate", "api_version|route_v2|sunset_header"),
   ("limitacion", "cuota|tasa|límite|429|estrangular", "quota|rate|limit|429|throttle", "rate_limit|retry_after|token_bucket")]),
 "observabilidad": (["sistema", "registro", "evento"], ["system", "log", "event"], [
   ("logs", "log|nivel|formato|rotar|mensaje", "log|level|format|rotate|message", "get_logger|set_level|rotate_logs"),
   ("metricas", "métrica|contador|histograma|panel|alerta", "metric|counter|histogram|dashboard|alert", "inc_counter|observe_hist|export_metrics"),
   ("trazas", "traza|span|latencia|distribuido|correlación", "trace|span|latency|distributed|correlation", "start_span|inject_trace|trace_id")]),
 "git": (["repositorio", "cambio", "rama"], ["repository", "change", "branch"], [
   ("ramas", "rama|checkout|puntero|crear|cambiar", "branch|checkout|pointer|create|switch", "git_branch|git_checkout|list_branches"),
   ("merge", "fusión|merge|conflicto|resolver|integrar", "merge|conflict|resolve|integrate|combine", "git_merge|resolve_conflict|abort_merge"),
   ("rebase", "rebase|historial|reescribir|reaplicar|linealizar", "rebase|history|rewrite|replay|linearize", "git_rebase|squash_commits|pick_commit")]),
 "ml": (["modelo", "entrenamiento", "datos"], ["model", "training", "data"], [
   ("gradiente", "gradiente|descenso|pérdida|paso|optimizar", "gradient|descent|loss|step|optimize", "grad_step|zero_grad|clip_norm"),
   ("sobreajuste", "sobreajuste|regularización|generalizar|memorizar|dropout", "overfitting|regularization|generalize|memorize|dropout", "add_dropout|l2_penalty|early_stop"),
   ("validacion", "validación|pliegue|cruzada|partición|estimar", "validation|fold|cross|split|estimate", "kfold_split|val_score|holdout")]),
 "red": (["red", "paquete", "conexión"], ["network", "packet", "connection"], [
   ("tcp", "tcp|segmento|retransmisión|ventana|fiable", "tcp|segment|retransmission|window|reliable", "tcp_connect|set_window|resend_segment"),
   ("dns", "dns|dominio|resolución|registro|nombre", "dns|domain|resolution|record|name", "resolve_host|lookup_a|flush_dns"),
   ("tls", "tls|certificado|cifrado|apretón|confianza", "tls|certificate|encryption|handshake|trust", "tls_handshake|load_cert|verify_chain")]),
 "memoria": (["memoria", "proceso", "objeto"], ["memory", "process", "object"], [
   ("gc", "recolector|basura|referencia|ciclo|liberar", "collector|garbage|reference|cycle|free", "gc_collect|weak_ref|ref_count"),
   ("virtual", "virtual|página|dirección|paginación|fallo", "virtual|page|address|paging|fault", "map_page|page_fault|swap_out"),
   ("fugas", "fuga|crecimiento|perfilar|heap|retención", "leak|growth|profile|heap|retention", "heap_snapshot|track_alloc|find_leak")]),
}

In [ ]:
import os, random, re

PLANT_ES = ["Esta guía explica cómo {a} influye en {b} cuando se trabaja con {c}.",
            "Para resolver el problema de {a}, revisa primero {b} y luego {c}.",
            "En producción conviene vigilar {a}, porque {b} cambia el comportamiento de {c}.",
            "Un error habitual es mezclar {a} con {b} sin entender {c}.",
            "La documentación recomienda medir {a} antes de tocar {b} o {c}.",
            "Ejemplo práctico: ajustar {a} reduce el coste de {b} en {c}."]
PLANT_EN = ["This guide explains how {a} affects {b} when working with {c}.",
            "To solve the problem with {a}, first check {b} and then {c}.",
            "In production you should watch {a}, because {b} changes the behavior of {c}.",
            "A common mistake is mixing {a} with {b} without understanding {c}.",
            "The documentation recommends measuring {a} before touching {b} or {c}.",
            "Practical example: tuning {a} lowers the cost of {b} in {c}."]
QS_ES = ["cómo gestionar {s}", "problemas con {s}", "guía de {s}", "qué hacer con {s}"]
QS_EN = ["how to handle {s}", "problems with {s}", "guide to {s}", "what to do about {s}"]

def _slot(rng, core, rare, shared, other):
    r = rng.random()
    return rng.choice(core) if r < .62 else rng.choice(shared) if r < .80 else rng.choice(rare) if r < .86 else rng.choice(other)

def genera_sintetico(n_temas: int = 8, docs_por_faceta: int = 4, seed: int = 42) -> dict:
    """Mismo esquema que el corpus real. Relevancia: docs de la misma faceta e idioma (code: faceta, queries 'code')."""
    rng = random.Random(seed)
    nombres = list(TEMAS)[:n_temas]
    docs, queries, qrels = [], [], {}
    for t in nombres:
        es_sh, en_sh, facetas = TEMAS[t]
        for fid, es, en, code in facetas:
            otros = [x for n in nombres if n != t for x in TEMAS[n][2][0][1].split("|")[:3]]
            otros_en = [x for n in nombres if n != t for x in TEMAS[n][2][0][2].split("|")[:3]]
            es, en, code = es.split("|"), en.split("|"), code.split("|")
            rel = {"es": {}, "en": {}, "code": {}}
            for lang, terms, shared, oth, plant in (("es", es, es_sh, otros, PLANT_ES), ("en", en, en_sh, otros_en, PLANT_EN)):
                for i in range(docs_por_faceta):
                    sents = [rng.choice(plant).format(a=_slot(rng, terms[:3], terms[3:], shared, oth),
                                                      b=_slot(rng, terms[:3], terms[3:], shared, oth),
                                                      c=_slot(rng, terms[:3], terms[3:], shared, oth)) for _ in range(3)]
                    did = f"{lang}-{t}.{fid}-{i}"
                    docs.append({"id": did, "lang": lang, "text": " ".join(sents)}); rel[lang][did] = 1
            for i in range(max(2, docs_por_faceta - 1)):
                fn = rng.choice(code); aux = rng.choice(code); w = rng.choice(en[:3])
                did = f"code-{t}.{fid}-{i}"
                docs.append({"id": did, "lang": "code", "text": f"def {fn}(x, n={rng.randint(2, 9)}):\n    \"\"\"{w} helper for {rng.choice(en_sh)}\"\"\"\n    return {aux}(x, {rng.choice(code)})"})
                rel["code"][did] = 1
            for lang, terms, qs in (("es", es, QS_ES), ("en", en, QS_EN), ("code", en, QS_EN)):
                s = " ".join(([rng.choice(terms[:3])] if rng.random() < .5 else []) + rng.sample(terms[3:], 2))   # 50%: 1 central + 2 raros
                qid = f"q-{lang}-{t}.{fid}"
                queries.append({"qid": qid, "lang": lang, "text": rng.choice(qs).format(s=s) + (" function" if lang == "code" else "")})
                qrels[qid] = dict(rel[lang])
    rng.shuffle(docs)
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": "synthetic"}

In [ ]:
def _carga_real(n_docs: int, seed: int = 42):
    """Cadena OFICIAL: HF `wikimedia/wikipedia` (ES/EN, streaming) + CodeSearchNet Python. Lanza excepción si no hay red/HF.
    Espejo: define HF_ENDPOINT=https://hf-mirror.com ANTES de importar `datasets` (reinicia el runtime) y repite."""
    from datasets import load_dataset
    docs, queries, qrels = [], [], {}
    for lang, cfg in (("es", "20231101.es"), ("en", "20231101.en")):
        ds = load_dataset("wikimedia/wikipedia", cfg, split="train", streaming=True).shuffle(seed=seed, buffer_size=2000)
        got = 0
        for row in ds:
            body = re.sub(r"\s+", " ", row["text"].split("\n\n")[0]).strip()
            if len(body) < 200:
                continue
            did, qid = f"{lang}-{row['id']}", f"q-{lang}-{row['id']}"
            docs.append({"id": did, "lang": lang, "text": body[:700]})
            queries.append({"qid": qid, "lang": lang, "text": row["title"]}); qrels[qid] = {did: 1}
            got += 1
            if got >= n_docs: break
    cs = load_dataset("code-search-net/code_search_net", "python", split="test", streaming=True)
    got = 0
    for i, row in enumerate(cs):
        doc_str = (row.get("func_documentation_string") or "").strip().split("\n")[0]
        code = row.get("func_code_string") or row.get("whole_func_string") or ""
        if len(doc_str) < 25 or not code: continue
        did, qid = f"code-{i}", f"q-code-{i}"
        docs.append({"id": did, "lang": "code", "text": code[:700]})
        queries.append({"qid": qid, "lang": "code", "text": doc_str}); qrels[qid] = {did: 1}
        got += 1
        if got >= n_docs: break
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": "real"}

def carga_corpus(fast_dev: bool = True) -> dict:
    """Contrato Políglota: {"docs":[{id,lang,text}], "queries":[{qid,lang,text}], "qrels":{qid:{doc_id:1}}, "source":"real"|"synthetic"}.
    Cadena: oficial (HF) -> espejo (HF_ENDPOINT) -> sintético. NOTA: la versión real usa queries = título del artículo /
    docstring (golden set auto-supervisado, más fácil que uno humano); el sintético usa facetas con huecos léxicos."""
    if os.environ.get("POLIGLOTA_FORCE_SYNTHETIC") != "1":
        try:
            return _carga_real(n_docs=150 if fast_dev else 1500)
        except Exception as e:  # sin red, proxy, dataset movido...
            print(f"⚠️ Datos reales no disponibles ({type(e).__name__}: {str(e)[:80]}) -> corpus SINTÉTICO.")
    return genera_sintetico(n_temas=8 if fast_dev else 12)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

def tok_simple(text: str) -> list[str]:
    """Token léxico: minúsculas + regex de palabras (snake_case queda como UN solo token; lo veremos fallar en código)."""
    return re.findall(r"\w+", text.lower())

def tok_code(text: str) -> list[str]:
    """Igual, pero parte snake_case y camelCase (set_ttl -> set, ttl). Es tokenización con criterio de dominio."""
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text).replace("_", " ")
    return re.findall(r"[^\W_]+", text.lower())

class BM25:
    """BM25 (Robertson/Lucene idf) con matriz dispersa. k1=1.2, b=0.75."""
    def __init__(self, tokenizer=tok_simple, k1=1.2, b=0.75):
        self.tok, self.k1, self.b = tokenizer, k1, b
    def fit(self, texts, ids):
        self.ids = list(ids)
        self.cv = CountVectorizer(tokenizer=self.tok, lowercase=False, token_pattern=None)
        X = self.cv.fit_transform(texts).tocsr().astype(float)
        self.N = X.shape[0]; self.dl = np.asarray(X.sum(1)).ravel(); self.avg = self.dl.mean()
        df = np.asarray((X > 0).sum(0)).ravel()
        self.idf = np.log(1 + (self.N - df + 0.5) / (df + 0.5))
        self.X = X.tocsc()
        return self
    def scores(self, query: str) -> np.ndarray:
        s = np.zeros(self.N)
        for j in set(self.cv.transform([query]).indices):
            col = self.X[:, j].toarray().ravel()
            s += self.idf[j] * col * (self.k1 + 1) / (col + self.k1 * (1 - self.b + self.b * self.dl / self.avg))
        return s

class LSI:
    """Retriever 'denso' sin descargas: TF-IDF sublineal + SVD truncada (LSA) + coseno. Hace de embedder de CPU."""
    def __init__(self, dim=48, seed=42):
        self.dim, self.seed = dim, seed
    def fit(self, texts, ids):
        self.ids = list(ids)
        self.tf = TfidfVectorizer(sublinear_tf=True)
        T = self.tf.fit_transform(texts)
        self.svd = TruncatedSVD(min(self.dim, T.shape[1] - 1), random_state=self.seed)
        self.Z = self._norm(self.svd.fit_transform(T))
        return self
    @staticmethod
    def _norm(Z): return Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-9)
    def scores(self, query: str) -> np.ndarray:
        return self.Z @ self._norm(self.svd.transform(self.tf.transform([query])))[0]

def rrf_scores(score_lists, k: int = 60) -> np.ndarray:
    """Reciprocal Rank Fusion: sum 1/(k + rango). Ignora la escala de cada score (ventaja y límite)."""
    out = np.zeros(len(score_lists[0]))
    for s in score_lists:
        order = np.lexsort((np.arange(len(s)), -np.asarray(s)))
        out[order] += 1.0 / (k + np.arange(len(s)) + 1)
    return out

def build_run(scorer, queries, doc_ids, topn: int = 100) -> dict:
    """scorer(texto_query) -> vector de scores alineado con doc_ids. Devuelve run {qid: [doc_ids top-n]}."""
    return {q["qid"]: rank_from_scores(scorer(q["text"]), doc_ids, topn) for q in queries}

In [ ]:
corpus = carga_corpus(fast_dev=FAST_DEV_RUN)
if corpus["source"] == "synthetic":
    corpus = genera_sintetico(n_temas=12, docs_por_faceta=40, seed=SEED)       # un SAE necesita miles de vectores (documentado)
elif FAST_DEV_RUN:
    corpus = carga_corpus(fast_dev=False)
docs = [d for d in corpus["docs"] if d["lang"] != "code"]; txt = [d["text"] for d in docs]; ids = [d["id"] for d in docs]
lang = np.array([d["lang"] for d in docs]); sint = corpus["source"] == "synthetic"
facet = np.array([d["id"].split("-", 1)[1].rsplit("-", 1)[0] for d in docs]) if sint else lang.copy()
topic = np.array([f.split(".")[0] for f in facet]) if sint else lang.copy()
fac_id = np.unique(facet, return_inverse=True)[1]
lsi = LSI(64, SEED).fit(txt, ids); E = lsi.Z.astype(np.float32)
print(f"fuente={corpus['source']} · {len(docs)} docs · {len(set(facet))} etiquetas de pureza · embeddings {E.shape}")

## 1. TopK SAE: la activación (TODO 1)
Te damos la clase `TopKSAE`, el entrenamiento (con inicialización *tied*, decoder normalizado y AuxK) y las utilidades. Falta lo esencial: la **activación TopK** en `encode`. Dado $x$, calcula las preactivaciones $\mathrm{pre}=W_{enc}(x-b_{dec})+b_{enc}$ (ya está en `self.pre`), conserva **solo las $k$ mayores por fila**, aplica ReLU a esos valores y devuelve `z` (con ceros en el resto) y `pre`.

<details><summary>🪜 Pista 1 (la idea)</summary>

`torch.topk(pre, k, dim=-1)` devuelve valores e índices. Crea `z` de ceros y coloca los valores en esos índices (`scatter`). El ReLU va sobre los valores seleccionados.
</details>

<details><summary>🪜 Pista 2 (casi la API)</summary>

`v, i = pre.topk(k or self.k, dim=-1)`; `z = torch.zeros_like(pre).scatter(-1, i, F.relu(v))`; `return z, pre`.
</details>

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

class TopKSAE(nn.Module):
    """SAE con activación TopK (Gao et al., OpenAI 2024; k-sparse AE de Makhzani & Frey 2013).
    z = TopK(W_enc (x - b_dec) + b_enc) ; x_hat = W_dec z + b_dec ; columnas de W_dec de norma 1. L0 = k exacto (sin λ que ajustar)."""
    def __init__(self, d: int, n_lat: int, k: int):
        super().__init__(); self.d, self.n, self.k = d, n_lat, k
        W = F.normalize(torch.randn(d, n_lat), dim=0)
        self.W_dec = nn.Parameter(W.clone()); self.W_enc = nn.Parameter(W.clone().T.contiguous())      # inicialización "tied" (transpuesta)
        self.b_enc = nn.Parameter(torch.zeros(n_lat)); self.b_dec = nn.Parameter(torch.zeros(d))
        self.register_buffer("pasos_sin_activar", torch.zeros(n_lat))
    def pre(self, x): return (x - self.b_dec) @ self.W_enc.T + self.b_enc
    def encode(self, x, k=None):
        """x: (B, d) -> (z, pre). z: (B, n_lat) dispersa con SOLO k valores no nulos (>= 0) por fila; pre: preactivaciones (B, n_lat)."""
        pre = self.pre(x)
        # TODO 1: quédate con las k mayores preactivaciones de cada fila (torch.topk), aplica ReLU a esos valores y
        #         colócalos en un tensor de ceros con scatter. Devuelve (z, pre).
        raise NotImplementedError("TODO 1: TopKSAE.encode")
    def decode(self, z): return z @ self.W_dec.T + self.b_dec
    def forward(self, x): z, pre = self.encode(x); return self.decode(z), z, pre
    @torch.no_grad()
    def normaliza_decoder(self): self.W_dec.data = F.normalize(self.W_dec.data, dim=0)

def entrena_topk(X, n_lat=256, k=4, pasos=1500, bs=256, lr=2e-3, auxk=True, k_aux=None, ventana_muerto=200, seed=0):
    """Entrena un TopK SAE. auxk=True añade la pérdida AuxK de Gao et al.: los latentes 'muertos' (sin activarse en `ventana_muerto`
    pasos) reconstruyen el RESIDUO del batch con sus top-k_aux preactivaciones -> los resucita. Devuelve (sae, historial)."""
    torch.manual_seed(seed); X = torch.as_tensor(X, dtype=torch.float32); d = X.shape[1]
    sae = TopKSAE(d, n_lat, k); sae.b_dec.data = X.mean(0).clone()                   # b_dec = media de los datos (centra la reconstrucción)
    opt = torch.optim.Adam(sae.parameters(), lr=lr); k_aux = k_aux or min(d // 2, n_lat // 4); var = ((X - X.mean(0)) ** 2).sum(1).mean()
    hist = {"nmse": [], "muertos": []}
    for t in range(pasos):
        x = X[torch.randint(0, len(X), (bs,))]; xh, z, pre = sae(x); err = x - xh
        loss = (err ** 2).sum(1).mean() / var
        activos = (z > 0).any(0); sae.pasos_sin_activar += 1; sae.pasos_sin_activar[activos] = 0
        muertos = sae.pasos_sin_activar > ventana_muerto
        if auxk and muertos.any():
            pm = pre.masked_fill(~muertos, -1e9); v, i = pm.topk(min(k_aux, int(muertos.sum())), dim=-1)
            za = torch.zeros_like(pre).scatter(-1, i, F.relu(v)); e_hat = za @ sae.W_dec.T
            loss = loss + (1 / 32) * ((err.detach() - e_hat) ** 2).sum(1).mean() / var
        opt.zero_grad(); loss.backward(); opt.step(); sae.normaliza_decoder()
        if t % 25 == 0: hist["nmse"].append(float(((err.detach()) ** 2).sum(1).mean() / var)); hist["muertos"].append(int(muertos.sum()))
    return sae.eval(), hist

def entrena_relu_l1(X, n_lat=256, lam=1e-2, pasos=1500, bs=256, lr=2e-3, seed=0):
    """SAE 'clásico' (Bricken et al., Anthropic 2023; Cunningham et al. 2023): z = ReLU(W_enc(x-b)+b_enc), pérdida = NMSE + λ·||z||_1."""
    torch.manual_seed(seed); X = torch.as_tensor(X, dtype=torch.float32); d = X.shape[1]
    sae = TopKSAE(d, n_lat, k=n_lat); sae.b_dec.data = X.mean(0).clone(); opt = torch.optim.Adam(sae.parameters(), lr=lr); var = ((X - X.mean(0)) ** 2).sum(1).mean()
    for _ in range(pasos):
        x = X[torch.randint(0, len(X), (bs,))]; z = F.relu(sae.pre(x)); xh = sae.decode(z)
        loss = ((x - xh) ** 2).sum(1).mean() / var + lam * z.abs().sum(1).mean()
        opt.zero_grad(); loss.backward(); opt.step(); sae.normaliza_decoder()
    sae.relu_mode = True; return sae.eval()

@torch.no_grad()
def codifica(sae, X, k=None):
    """Latentes (n, n_lat) y reconstrucción. Si el SAE es ReLU+L1 usa ReLU sin TopK."""
    X = torch.as_tensor(X, dtype=torch.float32)
    z = F.relu(sae.pre(X)) if getattr(sae, "relu_mode", False) else sae.encode(X, k)[0]
    return z, sae.decode(z)

def nmse(X, Xh):
    X, Xh = torch.as_tensor(X, dtype=torch.float32), torch.as_tensor(Xh, dtype=torch.float32)
    return float(((X - Xh) ** 2).sum() / ((X - X.mean(0)) ** 2).sum())

In [ ]:
def test_encode():
    torch.manual_seed(0); s = TopKSAE(8, 20, 3); x = torch.randn(5, 8); z, pre = s.encode(x)
    assert z.shape == (5, 20) and pre.shape == (5, 20) and (z >= 0).all(), "formas / no negatividad"
    assert ((z > 0).sum(1) <= 3).all() and (z > 0).sum() > 0, "como mucho k no nulos por fila"
    v, i = pre.topk(3, dim=-1); assert torch.allclose(z.gather(1, i), F.relu(v)), "los no nulos son los top-k (con ReLU)"
    z2, _ = s.encode(x, k=1); assert ((z2 > 0).sum(1) <= 1).all(), "el argumento k sobreescribe self.k"
    return "✅ test_encode OK"
def corre(f):
    try: print(f())
    except NotImplementedError as e: print(f"⏳ pendiente: {e}")
    except AssertionError as e: print(f"❌ falló: {e}")
    except Exception as e: print(f"❌ {type(e).__name__}: {e}")
corre(test_encode)

In [ ]:
pasos = 1500 if FAST_DEV_RUN else 4000
try:
    t0 = time.time(); sae, hist = entrena_topk(E, n_lat=96, k=4, pasos=pasos, auxk=True, seed=0); Z, Xh = codifica(sae, E); Z = Z.numpy()
    activos = (Z > 0).sum(0) >= 10; RES = {"nmse": nmse(E, Xh), "activos": int(activos.sum()), "l0": float((Z > 0).sum(1).mean())}
    print(f"entrenado en {time.time()-t0:.0f}s · NMSE={RES['nmse']:.3f} · L0={RES['l0']:.1f} · latentes activos {RES['activos']}/96")
except NotImplementedError as e:
    sae = Z = activos = None; RES = {}; print("⏳ pendiente:", e)

## 2. ¿Son interpretables las features? Pureza (TODO 2)
**Pureza** de una *feature* = fracción de sus **`top` documentos más activos** (por defecto 20) que comparten la **misma etiqueta** (faceta). Implementa `pureza_features(Z, etiquetas, top=20)` que devuelva un vector con la pureza de cada columna de `Z` (n_docs × n_features). Compararemos con las direcciones de **PCA** y los centroides de **k-means**.

<details><summary>🪜 Pista 1 (la idea)</summary>

Para cada columna: índices de los `top` mayores valores (`np.argsort(-col)[:top]`), cuenta etiquetas con `np.bincount` y divide el máximo por `top`.
</details>

<details><summary>🪜 Pista 2 (casi la API)</summary>

`np.array([np.bincount(etiquetas[np.argsort(-Z[:, j])[:top]]).max() / top for j in range(Z.shape[1])])` (etiquetas = enteros).
</details>

In [ ]:
def pureza_features(Z: np.ndarray, etiquetas: np.ndarray, top: int = 20) -> np.ndarray:
    # TODO 2
    raise NotImplementedError("TODO 2: pureza_features")

def test_pureza():
    Z = np.zeros((6, 2)); Z[:3, 0] = [3, 2, 1]; Z[3:, 1] = [3, 2, 1]; et = np.array([0, 0, 0, 1, 1, 0])
    p = pureza_features(Z, et, top=3); assert np.allclose(p, [1.0, 2 / 3]), f"pureza esperada [1, 2/3], obtenida {p}"
    return "✅ test_pureza OK"
corre(test_pureza)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
try:
    cols = np.where(activos)[0]; P = {"SAE TopK": pureza_features(Z[:, cols], fac_id)}
    n_f = len(cols); km = KMeans(n_f, n_init=3, random_state=0).fit(E); Ck = km.cluster_centers_
    P["k-means"] = pureza_features(E @ Ck.T / (np.linalg.norm(Ck, axis=1) + 1e-9), fac_id)
    pc = PCA(min(n_f, 64)).fit(E); P["PCA"] = pureza_features(np.abs((E - pc.mean_) @ pc.components_.T), fac_id)
    RES.update({"pureza_sae": float(P["SAE TopK"].mean()), "pureza_pca": float(P["PCA"].mean()), "pureza_km": float(P["k-means"].mean())})
    print({k: round(float(v.mean()), 3) for k, v in P.items()})
except (NotImplementedError, ValueError, TypeError, AttributeError, NameError) as e:
    print("⏳ pendiente (TODOs 1–2):", type(e).__name__)

## 3. Estabilidad entre semillas (TODO 3)
Entrena un segundo SAE con otra semilla. Implementa `estabilidad(W_a, W_b, umbral=0.9)`: para cada átomo de `W_a` (filas normalizadas, forma `(n_a, d)`) calcula el **coseno absoluto máximo** con los átomos de `W_b` y devuelve la **fracción con coseno > umbral**.

<details><summary>🪜 Pista 1 (la idea)</summary>

Matriz de cosenos `W_a @ W_b.T` (átomos unitarios), valor absoluto, máximo por fila, y media de `> umbral`.
</details>

<details><summary>🪜 Pista 2 (casi la API)</summary>

`C = np.abs(W_a @ W_b.T); return float((C.max(axis=1) > umbral).mean())`.
</details>

In [ ]:
def estabilidad(W_a: np.ndarray, W_b: np.ndarray, umbral: float = 0.9) -> float:
    # TODO 3
    raise NotImplementedError("TODO 3: estabilidad")

def test_estabilidad():
    A = np.eye(3); B = np.array([[1, 0, 0], [0, -1, 0], [0.6, 0.0, 0.8]]); assert abs(estabilidad(A, B) - 2 / 3) < 1e-9, "e1 y e2 (signo) coinciden; e3 no"
    return "✅ test_estabilidad OK"
corre(test_estabilidad)
try:
    sae_b, _ = entrena_topk(E, n_lat=96, k=4, pasos=pasos, auxk=True, seed=1); Zb, _ = codifica(sae_b, E); act_b = (Zb.numpy() > 0).sum(0) >= 10
    RES["estables"] = estabilidad(sae.W_dec.data.T.numpy()[activos], sae_b.W_dec.data.T.numpy()[act_b]); print(f"fracción de átomos estables entre semillas (cos>0.9): {RES['estables']:.2f}")
except (NotImplementedError, ValueError, TypeError, AttributeError, NameError) as e:
    print("⏳ pendiente (TODOs 1–3):", type(e).__name__)

## 4. Steering con una feature vs diferencia de medias (TODO 4)
Elige la *feature* más «git» (la de mayor fracción de documentos de tema «git» entre sus 20 más activos). Implementa `curva_steering(E, consultas, w, alphas, topic, objetivo)`: para cada $\alpha$ forma $q'=\mathrm{norm}(q+\alpha w)$ (con `q` normalizada), busca el top-10 por coseno (excluye el propio documento de la consulta) y devuelve la **fracción de resultados cuyo tema es `objetivo`**.

<details><summary>🪜 Pista 1 (la idea)</summary>

Normaliza `E` por filas; para cada α: `q = En[consultas] + α·w`, renormaliza, `sc = q @ En.T`, pon `-9` en la posición del propio documento y toma `argsort` descendente de los 10 primeros.
</details>

<details><summary>🪜 Pista 2 (casi la API)</summary>

`sc[np.arange(len(consultas)), consultas] = -9; top = np.argsort(-sc, axis=1)[:, :10]; float((topic[top] == objetivo).mean())`.
</details>

In [ ]:
def curva_steering(E, consultas, w, alphas, topic, objetivo):
    # TODO 4
    raise NotImplementedError("TODO 4: curva_steering")

def test_steering():
    r = np.random.default_rng(0); Et = np.concatenate([np.array([1, 0]) + 0.05 * r.standard_normal((10, 2)), np.array([0, 1]) + 0.05 * r.standard_normal((10, 2))]); tp = np.array(["a"] * 10 + ["b"] * 10)
    c = curva_steering(Et, np.array([0]), np.array([-1., 1.]), [0, 5], tp, "b"); assert c[0] <= 0.2 and c[1] >= 0.9, f"α=0 casi sin «b» y α=5 (dirección (−1,1)) casi todo «b»; obtenido {c}"
    return "✅ test_steering OK"
corre(test_steering)
try:
    En = E / np.linalg.norm(E, axis=1, keepdims=True); mu = {t: En[(topic == t) & (lang == "es")].mean(0) for t in ("cache", "git")}
    w_dm = mu["git"] - mu["cache"]; w_dm /= np.linalg.norm(w_dm); qs = np.where((topic == "cache") & (lang == "es"))[0][:100]
    j_git = max(cols, key=lambda j: np.mean(topic[np.argsort(-Z[:, j])[:20]] == "git")); w_sae = sae.W_dec.data[:, j_git].numpy(); alphas = [0, .5, 1, 1.5, 2, 3]
    RES["steer_sae"] = curva_steering(E, qs, w_sae, alphas, topic, "git"); RES["steer_dm"] = curva_steering(E, qs, w_dm, alphas, topic, "git")
    print("átomo SAE:", np.round(RES["steer_sae"], 2), "\ndif. medias:", np.round(RES["steer_dm"], 2))
except (NotImplementedError, ValueError, TypeError, AttributeError, NameError, KeyError) as e:
    print("⏳ pendiente (TODOs 1–4):", type(e).__name__)

## 5. Auditoría de filtración (TODO 5)
Como LSI es lineal, el atacante estima los términos del documento como $\hat x=z\,V$ (con $V$ = `lsi.svd.components_`). Implementa `precision_inversion(Z_emb, idx, n_top=10)`: para cada documento `i` de `idx`, toma los `n_top` términos con mayor $\hat x$ y devuelve la **fracción media que aparece realmente en el texto** del documento. Después mide el efecto de **ruido gaussiano** (σ ∈ {0, .05, .1, .2}) sobre inversión y sobre la utilidad (precisión@10 de faceta).

<details><summary>🪜 Pista 1 (la idea)</summary>

`est = Z_emb[i] @ Vk` (vector sobre el vocabulario), `np.argsort(-est)[:n_top]` son los índices de término; `inv_voc[j]` da la palabra; compara con `set(re.findall(r'\w+', txt[i].lower()))`.
</details>

<details><summary>🪜 Pista 2 (casi la API)</summary>

`np.mean([inv_voc[j] in palabras for j in top])` por documento y luego `np.mean` sobre `idx`.
</details>

In [ ]:
Vk = lsi.svd.components_; inv_voc = {j: t for t, j in lsi.tf.vocabulary_.items()}
def precision_inversion(Z_emb, idx, n_top=10):
    # TODO 5
    raise NotImplementedError("TODO 5: precision_inversion")

def test_inversion():
    p = precision_inversion(E, np.arange(20)); assert 0.5 < p <= 1.0, f"sin ruido debería recuperar mayormente términos reales (obtenido {p:.2f})"
    return "✅ test_inversion OK"
corre(test_inversion)
try:
    idx_i = np.random.default_rng(0).choice(len(txt), 150, replace=False); qi = np.random.default_rng(2).choice(len(E), 300, replace=False); rn = np.random.default_rng(1)
    RES["inv"], RES["util"] = {}, {}
    for s in (0, .05, .1, .2):
        Zn = E + s * rn.standard_normal(E.shape).astype(np.float32); Zn_n = Zn / np.linalg.norm(Zn, axis=1, keepdims=True); sc = E[qi] @ Zn_n.T; sc[np.arange(300), qi] = -9
        RES["inv"][s] = precision_inversion(Zn, idx_i); RES["util"][s] = float((facet[np.argsort(-sc, axis=1)[:, :10]] == facet[qi][:, None]).mean())
    print({s: (round(RES["inv"][s], 2), round(RES["util"][s], 2)) for s in RES["inv"]}, "(σ: (inversión, utilidad))")
except (NotImplementedError, ValueError, TypeError, AttributeError, NameError) as e:
    print("⏳ pendiente (TODO 5):", type(e).__name__)

## 6. Rúbrica (autoevaluación)
Umbrales **calibrados con la solución de referencia** (FAST_DEV_RUN=True, corpus sintético 12 temas, semilla 42) con el margen indicado. Con datos reales solo se exigen los chequeos estructurales.

In [ ]:
REF = {'nmse': 0.045, 'activos': 80, 'pureza_sae': 0.83, 'pureza_pca': 0.72, 'estables': 0.46, 'git_a0': 0.0, 'git_a2': 1.0}

def rubrica(res):
    out = []
    def chk(nombre, f):
        try: out.append((nombre, "✅" if f() else "❌"))
        except Exception: out.append((nombre, "⏳"))
    chk("1. test_encode", lambda: test_encode().startswith("✅")); chk("3. test_pureza", lambda: test_pureza().startswith("✅"))
    chk("4. test_estabilidad", lambda: test_estabilidad().startswith("✅")); chk("5. test_steering", lambda: test_steering().startswith("✅")); chk("6. test_inversion", lambda: test_inversion().startswith("✅"))
    if sint and FAST_DEV_RUN:
        chk(f"2a. NMSE ≤ 0.07 (ref {REF['nmse']})", lambda: res["nmse"] <= 0.07); chk("2b. L0 = 4", lambda: abs(res["l0"] - 4) < 1e-6); chk(f"2c. ≥ 60 latentes activos (ref {REF['activos']})", lambda: res["activos"] >= 60)
        chk(f"3a. pureza SAE ≥ 0.75 (ref {REF['pureza_sae']})", lambda: res["pureza_sae"] >= 0.75); chk("3b. pureza SAE > pureza PCA", lambda: res["pureza_sae"] > res["pureza_pca"])
        chk(f"4. estabilidad entre semillas = {REF['estables']} ± 0.25", lambda: abs(res["estables"] - REF["estables"]) <= 0.25)
        chk("5a. steering α=0 ≈ 0 % «git»", lambda: res["steer_dm"][0] < 0.1); chk("5b. steering diferencia de medias α=3 ≥ 90 %", lambda: res["steer_dm"][-1] >= 0.9)
        chk("5c. átomo SAE empuja hacia «git» (α=3 > α=0 + 0.2)", lambda: res["steer_sae"][-1] > res["steer_sae"][0] + 0.2)
        chk("6a. inversión sin ruido ≥ 0.8", lambda: res["inv"][0] >= 0.8); chk("6b. ruido σ=0.2 reduce la inversión ≥ 0.15", lambda: res["inv"][0] - res["inv"][0.2] >= 0.15)
        chk("6c. el ruido también reduce la utilidad (trade-off)", lambda: res["util"][0] - res["util"][0.2] >= 0.02)
    else: print("ℹ️ Datos reales o FAST_DEV_RUN=False: se omiten umbrales numéricos.")
    for n, e in out: print(e, n)
    print(f"\n{sum(e == '✅' for _, e in out)}/{len(out)} criterios cumplidos")
rubrica(RES)

## 7. Informe (escribe aquí)
1. **Calidad del SAE:** NMSE, L0 y latentes activos; ¿qué fracción de las features son interpretables (pureza ≥ 0.8)?
2. **Frente a las alternativas:** pureza vs k-means y PCA; steering vs diferencia de medias. ¿Qué aporta el SAE que k-means no?
3. **Fiabilidad:** estabilidad entre semillas. ¿Qué afirmaciones **no** harías sobre «lo que sabe el modelo»?
4. **Privacidad:** inversión sin ruido y con ruido; ¿qué política propones para Políglota (módulo 18)?

# ⛔ SPOILER — intenta resolverlo primero
Solución de referencia completa y ejecutable. Si te atascas en un TODO, abre antes las pistas 🪜 de ese paso. Al ejecutar se redefinen las funciones y se recalcula `RES`.

In [ ]:
def _encode_ref(self, x, k=None):
    pre = self.pre(x); v, i = pre.topk(k or self.k, dim=-1)
    return torch.zeros_like(pre).scatter(-1, i, F.relu(v)), pre
TopKSAE.encode = _encode_ref
corre(test_encode)
t0 = time.time(); sae, hist = entrena_topk(E, n_lat=96, k=4, pasos=pasos, auxk=True, seed=0); Z, Xh = codifica(sae, E); Z = Z.numpy()
activos = (Z > 0).sum(0) >= 10; RES = {"nmse": nmse(E, Xh), "activos": int(activos.sum()), "l0": float((Z > 0).sum(1).mean())}
print(f"entrenado en {time.time()-t0:.0f}s · NMSE={RES['nmse']:.3f} · L0={RES['l0']:.1f} · latentes activos {RES['activos']}/96")

In [ ]:
def pureza_features(Z, etiquetas, top=20):
    return np.array([np.bincount(etiquetas[np.argsort(-Z[:, j])[:top]]).max() / top for j in range(Z.shape[1])])
corre(test_pureza)
cols = np.where(activos)[0]; P = {"SAE TopK": pureza_features(Z[:, cols], fac_id)}; n_f = len(cols); km = KMeans(n_f, n_init=3, random_state=0).fit(E); Ck = km.cluster_centers_
P["k-means"] = pureza_features(E @ Ck.T / (np.linalg.norm(Ck, axis=1) + 1e-9), fac_id); pc = PCA(min(n_f, 64)).fit(E); P["PCA"] = pureza_features(np.abs((E - pc.mean_) @ pc.components_.T), fac_id)
RES.update({"pureza_sae": float(P["SAE TopK"].mean()), "pureza_pca": float(P["PCA"].mean()), "pureza_km": float(P["k-means"].mean())}); print({k: round(float(v.mean()), 3) for k, v in P.items()})

In [ ]:
def estabilidad(W_a, W_b, umbral=0.9):
    C = np.abs(W_a @ W_b.T); return float((C.max(axis=1) > umbral).mean())
corre(test_estabilidad)
sae_b, _ = entrena_topk(E, n_lat=96, k=4, pasos=pasos, auxk=True, seed=1); Zb, _ = codifica(sae_b, E); act_b = (Zb.numpy() > 0).sum(0) >= 10
RES["estables"] = estabilidad(sae.W_dec.data.T.numpy()[activos], sae_b.W_dec.data.T.numpy()[act_b]); print(f"fracción estable (cos>0.9): {RES['estables']:.2f}")

In [ ]:
def curva_steering(E, consultas, w, alphas, topic, objetivo):
    En = E / np.linalg.norm(E, axis=1, keepdims=True); out = []
    for a in alphas:
        q = En[consultas] + a * np.asarray(w); q /= np.linalg.norm(q, axis=1, keepdims=True); sc = q @ En.T; sc[np.arange(len(consultas)), consultas] = -9
        out.append(float((topic[np.argsort(-sc, axis=1)[:, :10]] == objetivo).mean()))
    return out
corre(test_steering)
En = E / np.linalg.norm(E, axis=1, keepdims=True); mu = {t: En[(topic == t) & (lang == "es")].mean(0) for t in ("cache", "git")}
w_dm = mu["git"] - mu["cache"]; w_dm /= np.linalg.norm(w_dm); qs = np.where((topic == "cache") & (lang == "es"))[0][:100]
j_git = max(cols, key=lambda j: np.mean(topic[np.argsort(-Z[:, j])[:20]] == "git")); w_sae = sae.W_dec.data[:, j_git].numpy(); alphas = [0, .5, 1, 1.5, 2, 3]
RES["steer_sae"] = curva_steering(E, qs, w_sae, alphas, topic, "git"); RES["steer_dm"] = curva_steering(E, qs, w_dm, alphas, topic, "git"); print("átomo SAE:", np.round(RES["steer_sae"], 2), "\ndif. medias:", np.round(RES["steer_dm"], 2))

In [ ]:
def precision_inversion(Z_emb, idx, n_top=10):
    out = []
    for i in idx:
        top = np.argsort(-(Z_emb[i] @ Vk))[:n_top]; palabras = set(re.findall(r"\w+", txt[i].lower())); out.append(np.mean([inv_voc[j] in palabras for j in top]))
    return float(np.mean(out))
corre(test_inversion)
idx_i = np.random.default_rng(0).choice(len(txt), 150, replace=False); qi = np.random.default_rng(2).choice(len(E), 300, replace=False); rn = np.random.default_rng(1); RES["inv"], RES["util"] = {}, {}
for s in (0, .05, .1, .2):
    Zn = E + s * rn.standard_normal(E.shape).astype(np.float32); Zn_n = Zn / np.linalg.norm(Zn, axis=1, keepdims=True); sc = E[qi] @ Zn_n.T; sc[np.arange(300), qi] = -9
    RES["inv"][s] = precision_inversion(Zn, idx_i); RES["util"][s] = float((facet[np.argsort(-sc, axis=1)[:, :10]] == facet[qi][:, None]).mean())
print({s: (round(RES["inv"][s], 2), round(RES["util"][s], 2)) for s in RES["inv"]}, "(σ: (inversión, utilidad))")
rubrica(RES)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6)); ax[0].bar(list(P), [v.mean() for v in P.values()], color=[VERDE, AZUL, GRIS]); ax[0].set_ylabel("pureza media"); ax[0].set_title("Interpretabilidad medida")
ax[1].plot(alphas, RES["steer_sae"], "o-", color=VERDE, label="átomo SAE"); ax[1].plot(alphas, RES["steer_dm"], "s--", color=AZUL, label="diferencia de medias"); ax[1].set_xlabel("α"); ax[1].set_ylabel("% «git» en el top-10"); ax[1].legend(frameon=False); ax[1].set_title("Steering")
s_ = list(RES["inv"]); ax[2].plot(s_, [RES["inv"][s] for s in s_], "o-", color=ROJO, label="inversión"); ax[2].plot(s_, [RES["util"][s] for s in s_], "s-", color=VERDE, label="utilidad"); ax[2].set_xlabel("σ del ruido"); ax[2].legend(frameon=False); ax[2].set_title("Privacidad vs utilidad"); ax[2].set_ylim(0, 1.02)
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** (1) Pureza: SAE > PCA, y k-means es competitivo o mejor: el SAE no «gana» en agrupar, sino en **descomponer** (4 átomos por documento). (2) Steering: la diferencia de medias iguala o supera al átomo SAE: no confundas legibilidad con control. (3) Privacidad: sin ruido la inversión es alta; el ruido la baja pero también daña la utilidad: **hay que elegir un punto de operación y medirlo contra un atacante real**.

## 🚀 Retos extra (nivel experto)
1. **Tamaño del diccionario:** entrena SAE con 48, 96, 384 latentes (mismo $k$) y mide pureza, *feature splitting* (¿una faceta se reparte en varias features?) y latentes muertos.
2. **JumpReLU:** implementa la variante con umbral aprendible (Rajamanoharan et al., 2024) y compara la frontera NMSE–$L_0$ con TopK.
3. **Auto-interpretabilidad con LLM** (celda opcional protegida con variable de entorno): pide a un LLM una etiqueta por feature a partir de sus 10 documentos más activos y **valida** con la pureza; reporta acuerdo.
4. **Embedder real:** repite con `paraphrase-multilingual-MiniLM-L12-v2` o `intfloat/multilingual-e5-small` (T4): ¿aparecen features multilingües (un átomo para ES y EN)? ¿Dimensiones dominantes (`diagnostico_dimensiones`)?
5. **LEACE / borrado:** proyecta fuera la dirección de idioma y mide con IC (módulo 15) si cambia el retrieval; compara con tu proyección de medias.
6. **Inversión real:** con `vec2text` (corrector para el embedder que uses) reconstruye 50 frases de Políglota y mide similitud/coincidencia exacta.
7. **Estabilidad con IC:** 5 semillas; reporta la fracción estable con IC bootstrap.

## 🧠 Reflexión
- ¿Qué decisión de producto tomarías con estas features (explicar resultados, auditar sesgos de idioma, dirigir el retrieval…)? ¿Cuál **no**?
- Si dos semillas dan features distintas, ¿cómo se protege un informe de interpretabilidad frente a «cherry-picking»?
- ¿Qué cambia en la auditoría de privacidad si Políglota indexa contratos con datos personales? (módulo 18)
- ¿Qué parte del banco de módulo 15 reutilizarías para evaluar la *utilidad* tras aplicar una defensa de ruido o cuantización?

➡️ **Siguiente:** módulo 18 (producción): versionado y migración de embedders, privacidad y monitorización.